In [ ]:
import h5py

with h5py.File(
    # "/home/magled/icicle-dev/results/pubchem_predictions.hdf5", "r"
    # "/home/magled/icicle-dev/results/eval/weighted_cosine_scaffold_s2/all_evaluation_spectra.hdf5",
    "/home/magled/icicle-dev/baselines/neims/outputs/neims_gnn_scaffold_v4_pool_bidir/predictions_scaffold_test.hdf5",
    "r",
) as f:
    print(f.keys())  # dataset names
    for key in list(f.keys()):  # print first 5 keys, which are metadata
        print(f[key].keys())  # attributes of each dataset
        # KeysViewHDF5 ['ground_truth_intensities', 'ground_truth_mz_bins', 'metrics', 'mz_bins', 'predicted_intensities', 'predicted_mz_bins']>
        break

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np

from icicle.utils.visualization.mass_spectra import (
    plot_mass_spectrum,
    plot_mirrored_spectra,
)
from icicle.utils.visualization.style import set_style

set_style()

with h5py.File(
    # "/home/magled/icicle-dev/results/pubchem_predictions.hdf5", "r"
    "/home/magled/icicle-dev/baselines/neims/outputs/neims_gnn_scaffold_v4_pool_bidir/predictions_scaffold_test.hdf5",
    # "/home/magled/icicle-dev/baselines/neims/results/predictions/neims_scaffold_s2_test.hdf5",
    "r",
) as f:
    keys = list(f.keys())[:10]

    for key in keys:
        predicted_intensities = f[key]["predicted_intensities"][:]
        # normalize s.t. max intensity is 1 for both spectra

        predicted_intensities /= np.max(predicted_intensities)
        predicted_mz_bins = f[key]["mz_bins"][:]

        # print(f[key]['metrics'])
        # <HDF5 group "/AABBMECPYUSEAW-UHFFFAOYSA-N/metrics" (0 members)> how to show the acrual values?
        plot_mass_spectrum(
            mz_values=predicted_mz_bins,
            intensities=predicted_intensities,
        )

In [ ]:
import h5py
import numpy as np

from icicle.utils.visualization.mass_spectra import (
    plot_mass_spectrum,
)
from icicle.utils.visualization.style import set_style

set_style()

with h5py.File(
    # "/home/magled/icicle-dev/results/pubchem_predictions.hdf5", "r"
    "/home/magled/icicle-dev/baselines/neims/outputs/neims_gnn_scaffold_fixed_offset_newhparams_from_ho/predictions_scaffold_test.hdf5",
    "r",
) as f:
    keys = list(f.keys())[:10]

    for key in keys:
        ground_truth_intensities = f[key]["ground_truth_intensities"][:]
        ground_truth_mz_bins = f[key]["ground_truth_mz_bins"][:]
        predicted_intensities = f[key]["predicted_intensities"][:]
        # normalize s.t. max intensity is 1 for both spectra
        ground_truth_intensities /= np.max(ground_truth_intensities)
        predicted_intensities /= np.max(predicted_intensities)
        predicted_mz_bins = f[key]["predicted_mz_bins"][:]

        # print(f[key]['metrics'])
        # <HDF5 group "/AABBMECPYUSEAW-UHFFFAOYSA-N/metrics" (0 members)> how to show the acrual values?
        plot_mirrored_spectra(
            ground_truth_intensities,
            predicted_intensities,
        )

In [ ]:
import h5py
import numpy as np

with h5py.File(
    # "/home/magled/icicle-dev/results/pubchem_predictions.hdf5", "r"
    "/home/magled/icicle-dev/results/eval/icicle_scaff_no_xeno_sim/all_evaluation_spectra.hdf5",
    "r",
) as f:
    smiles = f["smiles"][:]
    intensities = f["intensities"][:]

# m/z axis — bins are 0..749 by default (1 Da bins starting at min_mz)
# Check your model's min_mz/max_mz; default is typically 0-750
n_bins = intensities.shape[1]
mz = np.arange(n_bins)  # or np.linspace(min_mz, max_mz, n_bins)

from icicle.utils.visualization.style import set_style

set_style()

for i in range(10):
    plot_mass_spectrum(
        mz, intensities[i], title=smiles[i].decode(), smiles=smiles[i].decode()
    )

In [ ]:
import numpy as np
import pandas as pd

from icicle.data.fragmentation_engine import (
    FragmentationParams,
    FragmentEngine,
)

labels = pd.read_csv(
    "/home/magled/icicle-dev/data/NIST2023_GCMS_main/metadata.tsv", sep="\t"
)
counts = []
for smi in labels["standardized_smiles"].sample(10, random_state=42):
    try:
        e = FragmentEngine(
            mol_str=smi,
            params=FragmentationParams(
                max_tree_depth=3, max_broken_bonds=6, num_h_shifts=1
            ),
        )
        e.generate_fragments()
        counts.append(len(e.frag_to_entry))
    except:
        pass
print(pd.Series(counts).describe(percentiles=[0.5, 0.9, 0.95, 0.99]))
print("pct > 300:", (np.array(counts) > 300).mean())

In [ ]:
# results_dir = "/home/magled/icicle-dev/results/single_run/2025-07-26/19-45-44/"
results_dir = "/home/magled/icicle-dev/results/eval/icicle_scaff_no_xeno_sim/"

In [ ]:
metric_latex_labels = {
    "entropy_distance": r"d_{entr}",
    "cosine_similarity": r"s_{cos}",
    "composite_similarity_nist_gc": r"s_{comp}",
    "weighted_cosine_nist_gc": r"s_{wc}",
}

In [ ]:
import os

import h5py

from icicle.utils.visualization import set_style
from icicle.utils.visualization.mass_spectra import plot_mirrored_spectra

set_style()

with h5py.File(
    os.path.join(results_dir, "all_evaluation_spectra.hdf5"), "r"
) as f:
    for idx, key in enumerate(f.keys()):
        # print(f[key].keys())
        # 'ground_truth_intensities', 'ground_truth_mz_bins', 'metrics', 'predicted_intensities', 'predicted_mz_bins'
        pred_spec = (
            f[key]["predicted_intensities"][:]
            / f[key]["predicted_intensities"][:].max()
        )

        metrics_group = f[f"{key}/metrics"]

        smiles = f[key].attrs["smiles"]
        inchi_key = f[key].attrs["inchi_key"]

        cosine_similarity = round(metrics_group.attrs["cosine_similarity"], 3)
        composite_similarity = round(
            metrics_group.attrs["composite_similarity_nist_gc"], 3
        )
        entropy_distance = round(metrics_group.attrs["entropy_distance"], 3)
        weighted_cosine_nist_gc = round(
            metrics_group.attrs["weighted_cosine_nist_gc"], 3
        )

        title = f"{metric_latex_labels['cosine_similarity']} {cosine_similarity} - {metric_latex_labels['composite_similarity_nist_gc']} {composite_similarity} - {metric_latex_labels['entropy_distance']} {entropy_distance} - {metric_latex_labels['weighted_cosine_nist_gc']} {weighted_cosine_nist_gc}"

        # print(f[key]["predicted_intensities"][:])
        # print(f[key]["predicted_mz_bins"][:])
        plot_mirrored_spectra(
            true_spec=f[key]["ground_truth_intensities"][:],
            pred_spec=pred_spec,
            true_smiles=smiles,
            title=title,
            fade_unmatched=True,
        )

        if idx > 10:
            break

In [ ]:
import pandas as pd

df = pd.read_csv(os.path.join(results_dir, "similarity_results.csv"))
df.describe()

In [ ]:
import pandas as pd
import seaborn as sns

# # Method 1: Using subplots (separate violin plots)
metrics = [
    "entropy_distance",
    "cosine_similarity",
    "composite_similarity_nist_gc",
    "weighted_cosine_nist_gc",
]

# fig, axes = plt.subplots(1, len(metrics), figsize=(6,6))
# fig.suptitle('Distribution of Similarity Metrics', fontsize=16)

# for i, metric in enumerate(metrics):
#     sns.violinplot(y=df[metric], ax=axes[i])
#     axes[i].set_title(metric.replace('_', ' ').title())
#     axes[i].set_xlabel('')

# plt.tight_layout()
# plt.show()

# Method 2: Using melted data (all violin plots in one figure)
# Reshape data from wide to long format
df_melted = df[metrics].melt(var_name="Metric", value_name="Value")
# rename columns to latex labels
df_melted["Metric"] = df_melted["Metric"].map(metric_latex_labels)

plt.figure(figsize=(6, 4))
sns.violinplot(data=df_melted, x="Metric", y="Value")
plt.title("Distribution of Similarity Metrics")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Method 3: More customized version with better styling
plt.figure(figsize=(6, 4))
ax = sns.violinplot(data=df_melted, x="Metric", y="Value", palette="Set2")

# Customize the plot
plt.title("Distribution of Similarity Metrics", fontsize=16, fontweight="bold")
plt.xlabel("Metrics", fontsize=14)
plt.ylabel("Values", fontsize=14)
plt.xticks(rotation=45, ha="right")

# Add grid for better readability
plt.grid(True, alpha=0.3)

# Adjust layout
plt.tight_layout()
plt.show()

In [ ]:
# print SMILES of all spectra with entropy_distance < 0.2

smiles_to_plot = []

for idx, row in df.iterrows():
    if row["entropy_distance"] < 0.1:
        smiles_to_plot.append(row["smiles"])

print(smiles_to_plot)
print(len(smiles_to_plot))

In [ ]:
with h5py.File(
    results_dir + "/all_evaluation_spectra.hdf5",
    "r",
) as f:
    for key in f.keys():
        smiles = f[key].attrs["smiles"]
        if smiles in smiles_to_plot:
            fig = plot_mirrored_spectra(
                true_spec=f[key]["ground_truth_intensities"][:],
                pred_spec=f[key]["predicted_intensities"][:],
                true_smiles=smiles,
            )

            fig.savefig(f"mirrored_spectra_{smiles}.png")